# 共通論文の読解記録：朝と、説明し直した後

目的：AIを使わずに書いた自分の説明を保存する。準備の目安5分、読解時間は別。**このNotebookはAPIを呼びません。**
Colabでは次の準備セルのチェック欄をONにして実行します。コードを見る場合は「Show code」。
Colabはネット接続が必要。ローカルは準備済み環境を使います。

**スライドとの対応**：6「記録は手元に保存し、提出はPDFにする」、7「10分で読む：まず自分の説明を書く」。15:40の相互説明後も、このNotebookへ戻って自分の説明を残します。
朝はAIを開かず、分からない点もそのまま残します。午後はAI画面を閉じ、原論文を見ながら説明し直します。二つの段階は別の記録になり、朝の説明を上書きしません。
保存したMarkdownは自分が読むため、JSONLは全項目を再読込みするための形式です。


In [ ]:
# @title 0. 教材を準備する { display-mode: "form" }
from pathlib import Path
import sys
PREPARE_COLAB = False  # @param {type:"boolean"}
RELEASE_COMMIT = "6b7a8e27b4bafbb4520840b5c306af9eab34c9b0"
RELEASE_SHA256 = "a467a81511b18e0dddb941ee319c63fe74e3e223e051d1317862720d70844870"
BOOTSTRAP_SHA256 = "f1ad167aeec1cf95e2aa835eef73c05fdeae916445985ab69679ec6060d99914"
if PREPARE_COLAB:
    from urllib.request import urlopen
    from hashlib import sha256
    if not RELEASE_COMMIT or not RELEASE_SHA256:
        raise RuntimeError('この改訂版のColab公開準備はまだ完了していません。')
    url = f'https://raw.githubusercontent.com/humansys-lab/multimodal-ai-paper-reading/{RELEASE_COMMIT}/scripts/colab_bootstrap.py'
    with urlopen(url, timeout=40) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or sha256(source).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError('準備コードのハッシュが一致しません。実行を中止しました。')
    setup = {}
    exec(compile(source, 'verified_colab_bootstrap.py', 'exec'), setup)
    ROOT = setup['prepare_notebook'](RELEASE_COMMIT, RELEASE_SHA256, profile='records')
else:
    ROOT = globals().get('ROOT', Path.cwd()).resolve()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材フォルダで実行するか、ColabではPREPARE_COLABをTrueにしてください。')
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
print('教材フォルダ:', ROOT.name)

## 1. 記録する段階と、共通の論文・範囲・3問を確認する

朝は`record_stage='朝：AIなしで読む'`。相互説明後は`'午後：説明し直す'`へ変更して、このセルを実行します。論文・版・範囲・3問は同じです。


In [ ]:
from seminar_lab.config import load_yaml, material_context
from seminar_lab.records import new_record, load_records
from seminar_lab.ui import activity_description, save_pair
course = load_yaml(ROOT / 'config/course.yaml')
manifest = load_yaml(ROOT / 'materials/manifest.yaml')
record_stage = '朝：AIなしで読む'  # 午後は '午後：説明し直す'
stages = {'朝：AIなしで読む': 'P0', '午後：説明し直す': 'P3'}
if record_stage not in stages:
    raise ValueError('記録する段階は、朝か午後のどちらかを指定してください。')
activity_id = stages[record_stage]
print(activity_description(course, manifest, activity_id))
if 'manual_records' not in globals():
    manual_records = {}
if activity_id not in manual_records:
    context = material_context(course, manifest, activity_id, for_input=False)
    manual_records[activity_id] = new_record(context)
offline_record = manual_records[activity_id]


## 2. 自分の説明を書き、保存する

`None`は前の記入を保持します。自分の文を`'自分の説明'`のように囲んで記入し、`SAVE=True`にします。複数行なら`"""`で前後を囲めます。
朝・午後の両方を記入済みなら、二つの記録を同じファイルへ保存します。保存するたびに別の名前になり、古い保存ファイルも残ります。


In [ ]:
annotations = {
    'student_explanation': None,  # 自分の説明： '...' の中に記入
    'evidence_location': None,    # 根拠のページ・図・本文
    'unresolved_point': None,     # まだ分からないこと
}
SAVE = False
if SAVE:
    if any(x is not None and not isinstance(x, str) for x in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    offline_record.update({k: v for k, v in annotations.items() if v is not None})
    offline_record.update(status='manual', service='none', cost_kind='not_applicable')
    paths = save_pair(list(manual_records.values()), ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == list(manual_records.values())

## 閉じる前に

ColabのFiles欄で教材フォルダ内の`outputs`からJSONLとMarkdownをダウンロードします。再起動するとランタイム内のファイルやメモリの記録が失われます。ダウンロードした朝の記録は手元に残してください。
朝のAI利用は教員が案内するGUIで行います。APIで質問する際は[対話Notebook](01_dialogue_lab.ipynb)へ進みます。
